# TAMPO Algorithms — Colab Test Run Notebook

This notebook tests the currently implemented TAMPO meta-RL algorithms (TAMPO-GCN, TAMPO-GAT, and TAMPO-LSTM).
Run cells **top to bottom** in order.

## 0. Setup — Clone repo & install dependencies

In [ ]:
# ── 0a. Clone the repo ────────────────────────────────────────────
!git clone -b Vikkesh https://github.com/vikkesh/tampo.git tampo
%cd tampo

In [ ]:
# If you want to pull latest commits from the Vikkesh branch

# 1. Fetch the latest changes from GitHub
!git fetch origin

# 2. Checkout gat-pyg branch
!git checkout Vikkesh

# 3. Hard reset to exactly match the latest code on that branch
!git reset --hard origin/Vikkesh

In [ ]:
!nvidia-smi

In [2]:
# ── 0b. Install all dependencies ─────────────────────────────────
!pip install -r requirements.txt

In [3]:
# ── 0c. CUDA config + Verify imports ────────────────────────────────────────
# PYTORCH_CUDA_ALLOC_CONF must be set BEFORE CUDA operations start.
# Setting it here (same cell as first torch use) guarantees it survives
# any kernel restart triggered by pip install above.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch, gymnasium as gym, yaml, json, numpy as np
from torch_geometric.data import Batch, Data

print('torch           :', torch.__version__)
print('gymnasium       :', gym.__version__)
print('CUDA available  :', torch.cuda.is_available())
print('ALLOC_CONF      :', os.environ['PYTORCH_CUDA_ALLOC_CONF'])
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    free_gb  = torch.cuda.mem_get_info()[0] / 1024**3
    total_gb = props.total_memory / 1024**3
    print(f'GPU             : {props.name}')
    print(f'VRAM            : {total_gb:.1f} GB total | {free_gb:.1f} GB free')


## 1. Unit Test — DAGEncoder with Bi-GCN and Bi-GATv2 paths

Verifies that both encoders:
- Accept a PyG Batch of **variable-size** graphs
- Produce a context vector of the correct shape
- Produce **identical output shapes** (the only variable is the conv operator)

In [ ]:
import sys, os
sys.path.insert(0, '.')

import torch
from torch_geometric.data import Batch, Data
from algorithms.rl.tampo import DAGEncoder
from env.base_offloading_env import TaskOffloadingEnv

TASK_FEATURE_DIM   = TaskOffloadingEnv.TASK_FEATURE_DIM   # 9 (was 6 before the overhaul)
HIDDEN_DIM         = 128
SERVER_FEATURE_DIM = 20

# Three graphs with DIFFERENT node counts (8, 12, 20)
graphs = []
for n in [8, 12, 20]:
    x = torch.rand(n, TASK_FEATURE_DIM)
    src = torch.arange(0, n - 1)
    dst = torch.arange(1, n)
    edge_index = torch.stack([src, dst], dim=0)
    graphs.append(Data(x=x, edge_index=edge_index, num_nodes=n))

batch = Batch.from_data_list(graphs)
task_features_dummy = torch.zeros(3, 20, TASK_FEATURE_DIM)
server_features     = torch.rand(3, SERVER_FEATURE_DIM)

results = {}
for enc_type in ['gcn', 'gat']:
    encoder = DAGEncoder(
        task_feature_dim=TASK_FEATURE_DIM,
        hidden_dim=HIDDEN_DIM,
        num_layers=2,
        encoder_type=enc_type,
        server_feature_dim=SERVER_FEATURE_DIM
    )
    encoder.eval()
    with torch.no_grad():
        encoded_tasks, context = encoder(
            task_features=task_features_dummy,
            graph_batch=batch,
            server_features=server_features
        )
    assert context.shape       == (3, HIDDEN_DIM * 2), f"[{enc_type}] Bad context shape: {context.shape}"
    assert encoded_tasks.shape == (3, 20, HIDDEN_DIM * 2), f"[{enc_type}] Bad encoded_tasks shape: {encoded_tasks.shape}"
    assert not torch.isnan(context).any(),       f"[{enc_type}] NaN in context"
    assert not torch.isnan(encoded_tasks).any(), f"[{enc_type}] NaN in encoded_tasks"

    # ── THE critical invariant ────────────────────────────────────────────────
    # Before the 2026-07-10 overhaul the GNN mean-pooled the whole graph to a single
    # scalar and broadcast it to every node slot. encoded_tasks was non-zero (so the
    # old assertion passed) but every row was IDENTICAL, so the decoder's attention
    # ran over N copies of the same vector and no per-node information ever reached
    # the policy. Assert that real nodes now have distinct embeddings.
    g0 = encoded_tasks[0, :8]                       # graph 0 has 8 real nodes
    pairwise_spread = (g0.unsqueeze(0) - g0.unsqueeze(1)).abs().max().item()
    assert pairwise_spread > 1e-6, (
        f"[{enc_type}] All node embeddings are identical — the encoder is pooling away "
        f"per-node structure. The policy cannot distinguish one node from another."
    )

    # Padding slots (nodes 8..19 of graph 0) must be masked to zero
    assert encoded_tasks[0, 8:].abs().max().item() < 1e-6, f"[{enc_type}] padding not masked"

    results[enc_type] = {'context': context.shape, 'encoded_tasks': encoded_tasks.shape}
    print(f"PASS [{enc_type.upper()}] context={tuple(context.shape)}  encoded_tasks={tuple(encoded_tasks.shape)}  "
          f"per-node spread={pairwise_spread:.4f}  padding masked")

# Shape equality check — critical for fair comparison
assert results['gcn']['context']       == results['gat']['context'],       "MISMATCH: context shapes differ"
assert results['gcn']['encoded_tasks'] == results['gat']['encoded_tasks'], "MISMATCH: encoded_tasks shapes differ"
print("\nPASS - GCN and GAT produce identical output shapes, and nodes are distinguishable")


## 2. Generate the Golden Test Dataset

> ⚠️ Run **once only**.  Never re-run between algorithm comparisons.

In [5]:
!python utils/generate_test_dataset.py --num_dags 20 --output data/test_dags.json

import json
with open("data/test_dags.json") as f:
    ds = json.load(f)
print(f"Dataset contains {len(ds)} entries")
print("First entry keys:", list(ds[0].keys()))

## 2.5 Verification — Physics and Reward Overhaul

Confirms that the environment correctly implements dynamic server loads (Makespan) and diverse, context-sensitive rewards.

In [ ]:
# ── 2.5 Verification — Physics and Reward Overhaul ──────────────────────────
# Verifies the reward-signal fixes applied on 2026-07-10:
#   kappa       — 1e-27, so local energy sits between edge and cloud (was 1e-23,
#                 which made local ~6500x pricier than any offload action)
#   queue_wait  — congestion penalty measures the REAL queue wait
#   comm_time   — communication penalty no longer saturates at 1.0
#   r_delay/r_energy — per-objective components exposed in info, carrying penalties

import sys, yaml, json, numpy as np
sys.path.insert(0, '.')
from env.base_offloading_env import TaskOffloadingEnv

with open('configs/default_config.yaml') as f:
    fc = yaml.safe_load(f)
cfg = {}
for sec in ('system','computing','energy','network','tasks'):
    cfg.update(fc.get('environment', fc).get(sec, {}))
cfg['reward'] = fc.get('environment', fc).get('reward', {})

env = TaskOffloadingEnv(cfg)

with open('data/test_dags.json') as f:
    dags = json.load(f)
dag = dags[0]


def rollout(action_fn):
    """Run one full episode under a fixed policy; return aggregate stats."""
    env.reset(task_graph=dag, preference_vector=np.array([0.5, 0.5]))
    rew, rd, re_, qw, ct = [], [], [], [], []
    done, step = False, 0
    while not done and step < dag['num_tasks']:
        _, r, done, info = env.step(action_fn(step))
        rew.append(r); rd.append(info['r_delay']); re_.append(info['r_energy'])
        qw.append(info['queue_wait']); ct.append(info['comm_time'])
        step += 1
    return dict(reward=np.array(rew), r_delay=np.array(rd), r_energy=np.array(re_),
                queue_wait=np.array(qw), comm_time=np.array(ct),
                makespan=env.total_delay, energy=env.total_energy)


print('--- A: kappa calibration ---')
print(f'kappa in config: {env.kappa:.2e}  (should be 1e-27)')
assert 1e-28 <= env.kappa <= 1e-26, f'FAIL: kappa={env.kappa:.2e} outside the calibrated range'

# Real DAG nodes take `cycles` from the .gv expect_size attribute, NOT task_cycles_range.
cycles = np.median([t['cycles'] for t in dag['tasks']])
data   = np.median([t['data_size'] for t in dag['tasks']])
rate_c = env.bandwidth_up * np.log2(1 + env.cloud_power_tx / env.noise_power)
local_E = env.kappa * cycles * (env.local_freq ** 2)
cloud_E = (data * 1.05 / rate_c) * env.cloud_power_tx
print(f'median cycles={cycles:.3e}  local_E={local_E:.4f} J   cloud_E={cloud_E:.4f} J   ratio={local_E/cloud_E:.2f}x')
assert local_E / cloud_E < 10, f'FAIL: local is {local_E/cloud_E:.0f}x cloud — energy metric is on a cliff'
print('PASS — local energy is the same order as offload energy  ✓')

print('\n--- B: reward stays clipped to ±1 and components are exposed ---')
rr = rollout(lambda s: s % env.action_space.n)
for k in ('reward', 'r_delay', 'r_energy'):
    lo, hi = rr[k].min(), rr[k].max()
    assert lo >= -1.01 and hi <= 1.01, f'FAIL: {k} outside ±1 → [{lo:.3f}, {hi:.3f}]'
    print(f'{k:9s} range [{lo:+.3f}, {hi:+.3f}]  ✓')

print('\n--- C: congestion penalty measures a REAL queue wait ---')
# Piling every node onto one server must produce a larger mean queue wait
# than spreading them across the three edge servers.
stacked = rollout(lambda s: 1)              # everything on cloud
spread  = rollout(lambda s: 2 + (s % 3))    # across edge0/1/2
print(f'all-cloud   mean queue_wait = {stacked["queue_wait"].mean():.4f}s')
print(f'spread-edge mean queue_wait = {spread["queue_wait"].mean():.4f}s')
assert stacked['queue_wait'].mean() > 0, 'FAIL: queue_wait is identically zero — congestion penalty is dead'
assert stacked['queue_wait'].mean() > spread['queue_wait'].mean(), 'FAIL: stacking is not penalised more than spreading'
print('PASS — congestion penalty responds to server load  ✓')

print('\n--- D: communication penalty does not saturate ---')
# Penalties use the smooth relative overhead x/(x+local_delay), so they approach
# 1.0 but never pin there. Recompute it per node here to confirm it still varies.
rr_mixed = rollout(lambda s: s % env.action_space.n)   # forces cross-server edges
local_delays = np.array([t['cycles'] for t in dag['tasks']]) / env.local_freq
ct = rr_mixed['comm_time']
comm_pen = ct / (ct + local_delays[:len(ct)])
frac_saturated = float((comm_pen >= 0.999).mean())
print(f'comm_penalty mean={comm_pen.mean():.3f}  std={comm_pen.std():.3f}  pinned at 1.0 = {frac_saturated:.2f}')
assert frac_saturated == 0.0, 'FAIL: comm penalty pinned at 1.0 - it carries no gradient'
assert comm_pen.std() > 1e-3, 'FAIL: comm penalty is constant - it cannot discriminate placements'
print('PASS - communication penalty is informative  OK')

print('\n--- E: a real Pareto tension exists between cloud and edge ---')
cloud = rollout(lambda s: 1)
edge  = rollout(lambda s: 2)
loc   = rollout(lambda s: 0)
print(f'all-local  makespan={loc["makespan"]:.4f}s  energy={loc["energy"]:.4f}J')
print(f'all-cloud  makespan={cloud["makespan"]:.4f}s  energy={cloud["energy"]:.4f}J')
print(f'all-edge0  makespan={edge["makespan"]:.4f}s  energy={edge["energy"]:.4f}J')
assert cloud['makespan'] < edge['makespan'], 'FAIL: cloud should be faster than edge'
assert edge['energy']   < cloud['energy'],   'FAIL: edge should be cheaper than cloud'
print('PASS — cloud trades energy for speed; edge trades speed for energy  ✓')

print('\n--- F: server state dynamics ---')
env.reset(task_graph=dag, preference_vector=np.array([0.5, 0.5]))
snapshots = [env.server_available.copy()]
done, step = False, 0
while not done and step < dag['num_tasks']:
    _, _, done, _ = env.step(step % env.action_space.n)
    snapshots.append(env.server_available.copy())
    step += 1
all_same = all(np.allclose(snapshots[0], s) for s in snapshots[1:])
assert not all_same, 'FAIL — server state is static.'
print('PASS — server state changes dynamically  ✓')


## 3. Quick Training Smoke Test — TAMPO-GCN, TAMPO GAT and TAMPO-LSTM (1 iteration)

Checks that the full training loop executes without shape errors for both algorithms.

In [ ]:
import yaml, sys, os
sys.path.insert(0, '.')

with open('configs/default_config.yaml') as f:
    full_config = yaml.safe_load(f)

from env.base_offloading_env import TaskOffloadingEnv
from algorithms.rl.tampo import TAMPOFramework
from utils.training_setup import load_training_graphs, build_env_task_list
from utils.seeding import set_seed

SEED = full_config.get('experiment', {}).get('seed', 42)

cfg = {}
env_cfg = full_config.get('environment', full_config)
for sec in ('system','computing','energy','network','tasks'):
    cfg.update(env_cfg.get(sec, {}))
cfg['reward'] = env_cfg.get('reward', {})

# Deterministic pool: seed BEFORE load_training_graphs so the shuffle is identical every
# session and every encoder. DAGParser sorts files and takes the first N deterministically;
# the only randomness is the shuffle, which set_seed pins.
set_seed(SEED)
env = TaskOffloadingEnv(cfg)

# Pool: 20 graphs × 9 sizes = 180 total. Zero-shot generalisation is tested via UNSEEN
# graph topologies in test_dags.json — every graph there is a brand-new random DAG.
task_graphs   = load_training_graphs(verbose=False)
tasks_for_env = build_env_task_list(task_graphs)
env.load_task_dataset(tasks_for_env)

os.makedirs('models', exist_ok=True)

for enc_type in ['gcn', 'gat', 'lstm']:
    print(f'\n{"="*42}')
    print(f'  Smoke Test — TAMPO-{enc_type.upper()}')
    print(f'{"="*42}')
    tampo_cfg = {**full_config['training'], **full_config['algorithms']['tampo']}
    tampo_cfg['encoder_type'] = enc_type

    # seed passed to the constructor: seeding happens BEFORE weight init, so every encoder
    # starts from an identical RNG state and faces the identical task stream.
    framework = TAMPOFramework(env, tampo_cfg, seed=SEED)
    framework.train(num_iterations=1, meta_batch_size=2,
                    checkpoint_path=f'models/tampo_{enc_type}_smoke.pth')
    print(f'SMOKE TEST PASSED — TAMPO-{enc_type.upper()} ran one meta-iteration.')

## 3.5 Complete Training — TAMPO-GCN and TAMPO-LSTM

In [ ]:
# ── Persistence & experiment configuration ───────────────────────────────────
# Run this once per session, before the training cell below.
#
# Checkpoints are written DIRECTLY to CKPT_DIR every 10 iterations and at every clean
# stop. If that directory is on Google Drive, training survives a Colab session being
# killed: the next session mounts the same Drive folder and the training cell resumes
# from the exact iteration — and the exact RNG stream position — where it stopped.

import os

# Set False for a single long-running VM with its own disk (no Drive needed).
USE_DRIVE  = True
DRIVE_ROOT = '/content/drive/MyDrive/TAMPO_checkpoints'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')              # interactive auth popup, once per session
    CKPT_DIR = f'{DRIVE_ROOT}/models'
else:
    CKPT_DIR = 'models'                          # local disk (VM / Colab Pro long session)

os.makedirs(CKPT_DIR, exist_ok=True)

# Fail loudly if the checkpoint directory is not actually writable — otherwise a silent
# failure here would look like "starting fresh" and quietly discard a previous session.
_probe = os.path.join(CKPT_DIR, '.write_test')
try:
    with open(_probe, 'w') as _f:
        _f.write('ok')
    os.remove(_probe)
except OSError as e:
    raise RuntimeError(f'CKPT_DIR is not writable: {CKPT_DIR!r} ({e}). '
                       f'If USE_DRIVE, re-run the drive.mount step.')

existing = sorted(f for f in os.listdir(CKPT_DIR) if f.endswith('_checkpoint.pth'))
print(f'Checkpoints → {CKPT_DIR}')
print(f'Existing checkpoints: {existing if existing else "none (first session)"}')

In [ ]:
# ── Full Training Run — resumable, multi-encoder ─────────────────────────────
# Run the "Persistence & experiment configuration" cell above first (defines CKPT_DIR).
#
# HOW IT BEHAVES
#   • Pours this session's wall-clock budget into the encoders in ENCODERS order, one at a
#     time, each up to TARGET_ITERATIONS.
#   • Checkpoints (weights + optimizer + FULL RNG STREAM) are saved to CKPT_DIR every 10
#     iterations and at every clean stop.
#   • A resumed run is BIT-IDENTICAL to one continuous run: verified that 3+3 and 2+2+2
#     iterations reproduce 6 continuous iterations exactly. So splitting an encoder across
#     any number of sessions changes nothing.
#   • Every encoder is seeded identically at construction, so all three face the IDENTICAL
#     sequence of graphs, preferences and channel conditions. Only their weights and action
#     choices differ — which is the whole point of the comparison.
#
# STOPPING / SAVING MECHANISM
#   1. time_budget_s (per encoder = remaining session budget): train() stops at the next
#      iteration boundary once the budget is spent, saves, and returns. This is what keeps
#      a free-tier session from being killed mid-iteration with unsaved progress.
#   2. TARGET_ITERATIONS: an encoder already at the target is skipped.
#   3. 10-iteration autosave: even a hard kill loses at most the last <10 iterations.
#
# USAGE
#   • Free Colab (multiple sessions): leave ENCODERS = all three, SESSION_BUDGET_H ≈ 3.3.
#     Re-run this cell each session; it auto-advances gcn → gat → lstm as each finishes.
#     Keep going until every encoder prints "DONE".
#   • VM / Colab Pro (one long session): set SESSION_BUDGET_H high (e.g. 48). One run
#     trains all three to TARGET_ITERATIONS.
#   • Decide TARGET_ITERATIONS behaviourally: run the benchmark cell after each session;
#     stop when within_episode_entropy has risen off 0.000 and avg_makespan has plateaued
#     across two sessions. See docs/READING_RESULTS.md.

import os, time, yaml
from env.base_offloading_env import TaskOffloadingEnv
from algorithms.rl.tampo import TAMPOFramework
from utils.training_setup import load_training_graphs, build_env_task_list
from utils.seeding import set_seed

# ── experiment knobs ─────────────────────────────────────────────────────────
SEED              = 42
ENCODERS          = ['gcn', 'gat', 'lstm']   # fairness: every encoder must reach the SAME total
TARGET_ITERATIONS = 600                       # per encoder (behavioural stop is better; see above)
META_BATCH_SIZE   = 6                          # smaller batch = more optimiser steps per hour
EPISODES_PER_TASK = 3
SESSION_BUDGET_H  = 3.3                        # this session's wall-clock; raise for a VM

full_config = yaml.safe_load(open('configs/default_config.yaml'))
env_cfg = full_config['environment']
ecfg = {}
for sec in ('system','computing','energy','network','tasks'):
    ecfg.update(env_cfg[sec])
ecfg['reward'] = env_cfg['reward']

# Deterministic training pool (seed before the shuffle) — identical every session.
set_seed(SEED)
env = TaskOffloadingEnv(ecfg)
env.load_task_dataset(build_env_task_list(load_training_graphs(verbose=False)))

session_start = time.time()
session_budget_s = SESSION_BUDGET_H * 3600

for enc in ENCODERS:
    ckpt = f'{CKPT_DIR}/tampo_{enc}_checkpoint.pth'
    tcfg = {**full_config['training'], **full_config['algorithms']['tampo']}
    tcfg['encoder_type'] = enc
    tcfg['episodes_per_task'] = EPISODES_PER_TASK

    # seed → deterministic weight init; if a checkpoint exists, its RNG overrides the seed
    fw = TAMPOFramework(env, tcfg, model_path=ckpt, seed=SEED)
    done = fw.training_history['iterations']

    if done >= TARGET_ITERATIONS:
        print(f'\n✓ {enc.upper()} already at {done}/{TARGET_ITERATIONS} — skipping.')
        continue

    remaining = session_budget_s - (time.time() - session_start)
    if remaining <= 60:
        print(f'\n⏸ Session budget spent before reaching {enc.upper()}. '
              f'Re-run this cell in a new session to continue.')
        break

    print(f'\n### Training {enc.upper()}: {done} → up to {TARGET_ITERATIONS} '
          f'({remaining/3600:.2f} h left this session) ###')
    fw.train(
        num_iterations=TARGET_ITERATIONS - done,
        meta_batch_size=META_BATCH_SIZE,
        checkpoint_path=ckpt,          # writes straight to Drive if CKPT_DIR is on Drive
        time_budget_s=remaining,
    )
    fw.save(ckpt)

    now = fw.training_history['iterations']
    status = 'DONE.' if now >= TARGET_ITERATIONS else 'not finished — re-run to continue.'
    print(f'→ {enc.upper()}: {now}/{TARGET_ITERATIONS} iterations. {status}')

print('\nRun the benchmark cell to check within_episode_entropy and makespan.')

## 4. Run Benchmark on Trained Models

Compares both trained algorithms against the same test dataset.

In [ ]:
import os, sys
sys.path.insert(0, '.')

# Reads checkpoints from CKPT_DIR (defined in the "Persistence & experiment configuration"
# cell) — the same directory training wrote to, whether that is Google Drive or local disk.
CKPT_DIR = globals().get('CKPT_DIR', 'models')

# Which checkpoint does what:
#   tampo_<enc>_checkpoint.pth        RESUME checkpoint — weights + optimizer + RNG stream.
#                                     Used to continue training AND to benchmark.
#   tampo_<enc>_checkpoint_best.pth   lowest-meta-loss snapshot. NOTE: meta-loss is a weak
#                                     proxy for scheduling quality — prefer the plain
#                                     checkpoint unless you have a reason not to.

for enc_type in ['gcn', 'gat', 'lstm']:
    reg  = os.path.join(CKPT_DIR, f'tampo_{enc_type}_checkpoint.pth')
    best = os.path.join(CKPT_DIR, f'tampo_{enc_type}_checkpoint_best.pth')
    if os.path.exists(reg):
        print(f'[{enc_type.upper()}] will evaluate: {reg}')
    else:
        print(f'[{enc_type.upper()}] WARNING: no checkpoint at {reg} — untrained weights.')

print()

# Deterministic evaluation (fixed seeds, greedy actions, eval() mode → dropout off).
# Writes benchmark_results.csv, action_traces.csv, action_distribution.csv and plots.
# Watch the printed "within_episode_entropy": while it is 0.000 the policy is degenerate
# (one server per whole DAG) and the makespan/energy numbers do not reflect scheduling.
!python benchmark.py \
    --algorithms TAMPO_GCN TAMPO_GAT TAMPO_LSTM \
    --checkpoint_dir "{CKPT_DIR}" \
    --dataset_path data/test_dags.json \
    --output_dir results/ \
    --seed 42

## 5. Download Results

In [ ]:
import os
import glob
from google.colab import files

# Find the most recently created run directory
run_dirs = sorted(glob.glob('results/run_*'), reverse=True)
if run_dirs:
    latest_run = run_dirs[0]
    for fname in ['comparison_bar.png', 'pareto_front.png', 'benchmark_results.csv']:
        path = os.path.join(latest_run, fname)
        if os.path.exists(path):
            files.download(path)
            print(f'Downloaded {path}')
        else:
            print(f'Not found: {path}')
else:
    print('No results found. Run benchmark first.')


## 6. Download Models
Zip and download the trained model checkpoints.

In [ ]:
import shutil
from google.colab import files
import os

# Define the folder to zip and the output name
folder_to_zip = 'models'
output_filename = 'models.zip'

if os.path.exists(folder_to_zip):
    # Create zip archive
    shutil.make_archive('models', 'zip', folder_to_zip)
    
    # Trigger download
    files.download(output_filename)
    print(f'Successfully zipped and started download for {output_filename}')
else:
    print(f'Error: Folder "{folder_to_zip}" not found.')

## Backing up results andmodels folders to Gdrive

In [ ]:
# ── 8. Backup Results to Google Drive (WSL/IDE Fallback) ───────
from google.colab import drive
import shutil
import os
from datetime import datetime

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define destination with timestamp parent folder
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
drive_destination = f'/content/drive/MyDrive/Task_Offloading_Results/backup_{timestamp}'
os.makedirs(drive_destination, exist_ok=True)

# 3. Copy results and models folders
shutil.copytree('results', f'{drive_destination}/results', dirs_exist_ok=True)
shutil.copytree('models', f'{drive_destination}/models', dirs_exist_ok=True)
print(f'✅ Successfully backed up results and models to:{drive_destination}')